# XFeat-SuperPoint Hybrid Model v2 — Colab Training Notebook

**Run cells top-to-bottom.** Every cell is idempotent — safe to re-run after disconnects.

### Before you start
1. **Runtime → Change runtime type → GPU** (T4 is fine; A100 is faster).
2. Your MegaDepth data must live on **Google Drive** at a path like:
   ```
   My Drive/hybrid_feature_matching/data/MegaDepth/
   ├── 0001/dense0/{imgs,depths}/
   ├── 0002/dense0/{imgs,depths}/
   └── 0003/dense0/{imgs,depths}/
   ```
   Set `GDRIVE_MEGADEPTH_PATH` in **Cell 3** to wherever that folder lives on your Drive.

### Scene configuration (smoke-test vs. full training)
| Goal | `TRAIN_SCENES` | `VAL_SCENES` |
|---|---|---|
| 3-scene smoke-test (default) | `["0001","0002"]` | `["0003"]` |
| Full training | list all scene IDs | hold-out a few |

To train on all scenes change `USE_ALL_SCENES = True` in Cell 3.

## Cell 1 — GPU / runtime sanity check

In [ ]:
import os, sys, torch

colab_env = os.environ.get('COLAB_GPU') or os.environ.get('COLAB_RELEASE_TAG')
print('Running in Colab:', bool(colab_env))
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM (GB)       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('\n⚠️  No GPU detected — training will be very slow on CPU.')
    print('   Go to Runtime → Change runtime type → GPU and reconnect.')

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)

## Cell 2 — Mount Google Drive

A browser popup will ask you to authorise access. Click **Allow**.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('Drive mounted. Contents of /content/drive/MyDrive:')
!ls /content/drive/MyDrive | head -20

## Cell 3 — Configuration

**Edit the variables in this cell to match your setup.**

In [ ]:
import os
from pathlib import Path

# ── Repo + dependency pins ────────────────────────────────────────────────────
HYBRID_REPO_URL          = 'https://github.com/MalharRane/XFeat-SuperPoint-Hybrid-Model.git'
HYBRID_REPO_BRANCH       = ''  # branch or commit SHA; '' = default (main) branch
ACCELERATED_FEATURES_REF = 'e92685f57f8318b18725c5c8c0bd28c7fe188d9a'
SUPERPOINT_REPO_URL      = 'https://github.com/rpautrat/SuperPoint.git'
SUPERPOINT_REF           = '1411bbd68c50163555d39c1b26e9e046ebd48f27'

# ── Data paths ────────────────────────────────────────────────────────────────
# Path to MegaDepth root folder on your Google Drive.
# Adjust to wherever you stored it, e.g.:
#   '/content/drive/MyDrive/hybrid_feature_matching/data/MegaDepth'
GDRIVE_MEGADEPTH_PATH = '/content/drive/MyDrive/hybrid_feature_matching/data/MegaDepth'

# Where the dataset will be symlinked inside the Colab VM (no copying needed).
DATA_ROOT = '/content/MegaDepth'

# ── Scene selection ───────────────────────────────────────────────────────────
# Set USE_ALL_SCENES = True to auto-discover every scene in GDRIVE_MEGADEPTH_PATH.
# Set USE_ALL_SCENES = False to use only the explicit lists below (smoke-test).
USE_ALL_SCENES = False

# Used only when USE_ALL_SCENES = False:
SMOKE_TRAIN_SCENES = ['0001', '0002']
SMOKE_VAL_SCENES   = ['0003']

# ── Training hyperparameters (override config.yaml defaults) ──────────────────
MAX_EPOCHS         = 40      # reduce for quick tests
BATCH_SIZE         = 4       # reduce to 2 if OOM on T4
MAX_PAIRS_SCENE    = 1000    # pairs sampled per scene per epoch
VAL_MAX_BATCHES    = 20      # validation batches per epoch
RESUME_CHECKPOINT  = ''      # path to .pth file to resume from; '' = start fresh

# ── Logging backend ───────────────────────────────────────────────────────────
LOGGING_BACKEND = 'tensorboard'   # 'tensorboard' | 'wandb'

# ── Derived paths (do not edit) ───────────────────────────────────────────────
HYBRID_ROOT   = '/content/XFeat-SuperPoint-Hybrid-Model'
WEIGHTS_DIR   = f'{HYBRID_ROOT}/weights'
CKPT_DIR      = f'{HYBRID_ROOT}/hybrid_model_v2/checkpoints'
CONFIG_PATH   = f'{HYBRID_ROOT}/hybrid_model_v2/config.yaml'

print('Config summary')
print('  GDRIVE_MEGADEPTH_PATH :', GDRIVE_MEGADEPTH_PATH)
print('  DATA_ROOT             :', DATA_ROOT)
print('  USE_ALL_SCENES        :', USE_ALL_SCENES)
if not USE_ALL_SCENES:
    print('  TRAIN_SCENES          :', SMOKE_TRAIN_SCENES)
    print('  VAL_SCENES            :', SMOKE_VAL_SCENES)
print('  MAX_EPOCHS            :', MAX_EPOCHS)
print('  BATCH_SIZE            :', BATCH_SIZE)

## Cell 4 — Install missing packages (fast, no conflicts)

Colab already ships with PyTorch, NumPy, SciPy, Pillow, OpenCV, h5py, TensorBoard and tqdm at compatible versions.  
We only install the handful of packages that are **genuinely absent** from the base image, so we never trigger the version-conflict cascade seen when reinstalling PyTorch or NumPy.

In [ ]:
# Install only packages Colab does not include by default.
# ─────────────────────────────────────────────────────────
# kornia           : used by the dataset augmentation pipeline
# lightning-utilities: kornia runtime dependency
# pyyaml           : config loading (usually present; pinned for safety)
# h5py             : depth .h5 file reading (usually present; pinned for safety)
# ─────────────────────────────────────────────────────────
# We intentionally do NOT reinstall: torch, torchvision, numpy, scipy,
# Pillow, opencv, tqdm, tensorboard — they are already in Colab and
# reinstalling them causes the dependency-resolver conflicts reported earlier.

!pip install -q \
    kornia==0.7.4 \
    lightning-utilities==0.11.7 \
    pyyaml==6.0.2 \
    h5py==3.11.0

# Verify critical imports work
import torch, torchvision, cv2, h5py, yaml, kornia
print('torch      :', torch.__version__)
print('torchvision:', torchvision.__version__)
print('cv2        :', cv2.__version__)
print('h5py       :', h5py.__version__)
print('kornia     :', kornia.__version__)
print('yaml       :', yaml.__version__)
print('All critical imports OK ✓')

## Cell 5 — Clone repositories

In [ ]:
%cd /content

# ── 1. Hybrid model repo ──────────────────────────────────────────────────────
!rm -rf XFeat-SuperPoint-Hybrid-Model
!git clone --quiet {HYBRID_REPO_URL}
%cd /content/XFeat-SuperPoint-Hybrid-Model
if HYBRID_REPO_BRANCH:
    !git checkout --quiet {HYBRID_REPO_BRANCH}
print('Hybrid repo branch:', end=' ')
!git rev-parse --abbrev-ref HEAD
print('Commit:', end=' ')
!git rev-parse --short HEAD

# ── 2. accelerated_features (XFeat) ──────────────────────────────────────────
%cd /content
!rm -rf accelerated_features
!git clone --quiet https://github.com/verlab/accelerated_features.git
%cd /content/accelerated_features
!git checkout --quiet {ACCELERATED_FEATURES_REF}
print('XFeat commit:', end=' ')
!git rev-parse --short HEAD

# ── 3. SuperPoint ─────────────────────────────────────────────────────────────
%cd /content
!rm -rf SuperPoint
!git clone --quiet {SUPERPOINT_REPO_URL}
%cd /content/SuperPoint
!git checkout --quiet {SUPERPOINT_REF}
print('SuperPoint commit:', end=' ')
!git rev-parse --short HEAD

print('\nAll repos cloned ✓')

## Cell 6 — Set PYTHONPATH and download model weights

In [ ]:
import os, sys

# ── PYTHONPATH so that `modules.xfeat`, `superpoint.*` are importable ─────────
_paths = [
    HYBRID_ROOT,
    '/content/accelerated_features',
    '/content/SuperPoint',
]
os.environ['PYTHONPATH'] = ':'.join(_paths + [os.environ.get('PYTHONPATH', '')])
for p in reversed(_paths):
    if p not in sys.path:
        sys.path.insert(0, p)

print('PYTHONPATH (first 3):', os.environ['PYTHONPATH'].split(':')[:3])

# ── Download pre-trained weights ──────────────────────────────────────────────
%cd {HYBRID_ROOT}
!mkdir -p weights

import urllib.request
from pathlib import Path

XFEAT_URL = 'https://github.com/verlab/accelerated_features/releases/download/v1.0/xfeat.pt'
SP_URL    = 'https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth'

def _download_if_missing(url: str, dest: str):
    p = Path(dest)
    if p.exists() and p.stat().st_size > 0:
        print(f'  {p.name} — already present, skipping.')
        return
    print(f'  Downloading {p.name} ...')
    urllib.request.urlretrieve(url, dest)
    print(f'  {p.name} — done ({p.stat().st_size // 1024} KB)')

print('Weights:')
_download_if_missing(XFEAT_URL, f'{WEIGHTS_DIR}/xfeat.pt')
_download_if_missing(SP_URL,    f'{WEIGHTS_DIR}/superpoint_v1.pth')

# Quick import smoke-test
from modules.xfeat import XFeat
print('XFeat import       : OK ✓')
try:
    from superpoint.superpoint import SuperPoint
    print('SuperPoint import  : OK ✓ (superpoint.superpoint)')
except ImportError:
    try:
        from superpoint import SuperPoint
        print('SuperPoint import  : OK ✓ (superpoint)')
    except ImportError as e:
        print('SuperPoint import  : FAILED —', e)
        raise

## Cell 7 — Link dataset and verify scene layout

In [ ]:
import os
from pathlib import Path

# ── Symlink Drive folder into the VM's /content so train.py can use a short path
gdrive_path = Path(GDRIVE_MEGADEPTH_PATH)
data_root   = Path(DATA_ROOT)

if not gdrive_path.exists():
    raise FileNotFoundError(
        f"MegaDepth not found on Drive at: {gdrive_path}\n"
        "Check that Google Drive is mounted (Cell 2) and that "
        "GDRIVE_MEGADEPTH_PATH in Cell 3 is correct."
    )

if data_root.exists() or data_root.is_symlink():
    if data_root.is_symlink():
        os.remove(str(data_root))

if not data_root.exists():
    os.symlink(str(gdrive_path), str(data_root))
    print(f'Symlink created: {data_root} → {gdrive_path}')
else:
    print(f'{data_root} already exists (real directory); using as-is.')

# ── Discover all available scenes ─────────────────────────────────────────────
all_scenes = sorted([
    p.parent.parent.name
    for p in Path(DATA_ROOT).glob('*/dense0/imgs')
    if (p.parent / 'depths').exists()
])

print(f'\nDiscovered {len(all_scenes)} scenes in {DATA_ROOT}:')
print(all_scenes)

# ── Determine train/val split ─────────────────────────────────────────────────
if USE_ALL_SCENES:
    if len(all_scenes) < 2:
        raise RuntimeError('Need at least 2 scenes for train/val split.')
    val_count   = max(1, len(all_scenes) // 5)   # ~20% for val
    TRAIN_SCENES = all_scenes[:-val_count]
    VAL_SCENES   = all_scenes[-val_count:]
    print(f'\nAll-scenes mode: {len(TRAIN_SCENES)} train, {len(VAL_SCENES)} val')
else:
    TRAIN_SCENES = SMOKE_TRAIN_SCENES
    VAL_SCENES   = SMOKE_VAL_SCENES
    print(f'\nSmoke-test mode: train={TRAIN_SCENES}, val={VAL_SCENES}')

# ── Per-scene health check ────────────────────────────────────────────────────
missing = []
for scene in TRAIN_SCENES + VAL_SCENES:
    imgs   = Path(DATA_ROOT) / scene / 'dense0' / 'imgs'
    depths = Path(DATA_ROOT) / scene / 'dense0' / 'depths'
    n_imgs   = len(list(imgs.glob('*')))   if imgs.exists()   else 0
    n_depths = len(list(depths.glob('*'))) if depths.exists() else 0
    status = '✓' if (imgs.exists() and depths.exists() and n_imgs > 0) else '✗'
    print(f'  {status} {scene:>6}  imgs={n_imgs:>4}  depths={n_depths:>4}')
    if status == '✗':
        missing.append(scene)

if missing:
    raise RuntimeError(f'The following scenes are missing or empty: {missing}')
print('\nDataset check passed ✓')

## Cell 8 — Write runtime config and start training

This cell writes a temporary YAML that patches `hybrid_model_v2/config.yaml` with the scene lists and hyperparameters you set in Cell 3, then launches training.

In [ ]:
import yaml, os
from pathlib import Path

# ── Load base config ──────────────────────────────────────────────────────────
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# ── Apply runtime overrides ───────────────────────────────────────────────────
cfg['data_root']             = DATA_ROOT
cfg['train_scenes']          = TRAIN_SCENES
cfg['val_scenes']            = VAL_SCENES
cfg['max_epochs']            = MAX_EPOCHS
cfg['batch_size']            = BATCH_SIZE
cfg['max_pairs_per_scene']   = MAX_PAIRS_SCENE
cfg['val_max_batches']       = VAL_MAX_BATCHES
cfg['num_workers']           = 0          # must be 0 on Colab
cfg['logging_backend']       = LOGGING_BACKEND
cfg['superpoint_weights_path'] = f'{WEIGHTS_DIR}/superpoint_v1.pth'
cfg['xfeat_weights_path']    = f'{WEIGHTS_DIR}/xfeat.pt'
cfg['checkpoint_dir']        = CKPT_DIR
cfg['log_dir']               = f'{HYBRID_ROOT}/hybrid_model_v2/runs'
cfg['mixed_precision']       = True  # auto-disabled if no GPU or AMP fails

# ── Write patched config to /tmp so we don't dirty the repo ──────────────────
runtime_cfg_path = '/tmp/hybrid_v2_runtime.yaml'
with open(runtime_cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, default_flow_style=False)
print('Runtime config written to', runtime_cfg_path)

# ── Build CLI command ─────────────────────────────────────────────────────────
resume_flag = f'--resume {RESUME_CHECKPOINT}' if RESUME_CHECKPOINT else ''

train_cmd = (
    f'python {HYBRID_ROOT}/hybrid_model_v2/train.py '
    f'--config {runtime_cfg_path} '
    f'{resume_flag}'
)
print('Launching:', train_cmd)

# ── Run training ──────────────────────────────────────────────────────────────
os.chdir(HYBRID_ROOT)
!{train_cmd}

## Cell 9 — (Optional) Resume training from a checkpoint

Set `RESUME_CHECKPOINT` in Cell 3 to a `.pth` path and re-run Cell 8,  
**or** run this standalone cell to resume from the latest `best.pth`.

In [ ]:
from pathlib import Path
import os

best_ckpt = Path(CKPT_DIR) / 'best.pth'
if not best_ckpt.exists():
    print(f'No best.pth found at {best_ckpt}. Run Cell 8 first.')
else:
    print(f'Resuming from: {best_ckpt}')
    os.chdir(HYBRID_ROOT)
    !python {HYBRID_ROOT}/hybrid_model_v2/train.py \
        --config /tmp/hybrid_v2_runtime.yaml \
        --resume {best_ckpt}

## Cell 10 — (Optional) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {HYBRID_ROOT}/hybrid_model_v2/runs

## Cell 11 — (Optional) Copy best checkpoint back to Google Drive

In [ ]:
import shutil
from pathlib import Path

SAVE_TO_DRIVE = '/content/drive/MyDrive/hybrid_feature_matching/checkpoints'

src = Path(CKPT_DIR) / 'best.pth'
dst_dir = Path(SAVE_TO_DRIVE)

if not src.exists():
    print(f'best.pth not found at {src}. Complete at least one training epoch first.')
else:
    dst_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(src), str(dst_dir / 'best.pth'))
    print(f'Saved to {dst_dir / "best.pth"}')